# 5.14 — Defining Feature Columns and Target Variables

This notebook demonstrates how this repo defines **X (features)** and **y (target)** explicitly and safely:

- Target is defined clearly (what we predict).
- Features are defined intentionally (what we are allowed to use).
- Leakage columns are excluded (what we must NOT use).
- `X` and `y` are separated before fitting.

Golden rule: **a feature must exist at prediction time**.

## 1) Feature/target configuration
Definitions live in `src/config.py` for reviewability and reproducibility.

In [ ]:
from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
    TARGET_SOURCE_COLUMN,
)

print('TARGET_COLUMN:', TARGET_COLUMN)
print('TARGET_SOURCE_COLUMN:', TARGET_SOURCE_COLUMN)
print('NUMERICAL_FEATURES:', NUMERICAL_FEATURES)
print('CATEGORICAL_FEATURES:', CATEGORICAL_FEATURES)
print('EXCLUDED_COLUMNS:', EXCLUDED_COLUMNS)
print('ALL_FEATURES:', ALL_FEATURES)

## 2) Load raw data (no transformations)
The loader reads the CSV only. It does not create labels or transform features.

In [ ]:
from src.data_loader import load_csv

df = load_csv('data/raw/source_demo_crops_20260321.csv')
df.head()

## 3) Define y (target) and X (features) explicitly

For this demo dataset we **derive** a binary label from `yield_kg` (median split) and call it `target`.
Then we select features from the explicit feature lists.

In [ ]:
from src.preprocessing import separate_features_and_target

df_work = df.copy()
df_work[TARGET_COLUMN] = (df_work[TARGET_SOURCE_COLUMN] >= df_work[TARGET_SOURCE_COLUMN].median()).astype(int)

available_features = [c for c in ALL_FEATURES if c in df_work.columns]
X, y = separate_features_and_target(
    df_work,
    target_column=TARGET_COLUMN,
    feature_columns=available_features,
    excluded_columns=EXCLUDED_COLUMNS,
    verbose=True,
)

X.head(), y.head()

## 4) Training uses X and y and fits preprocessing on train only

The training pipeline fits preprocessing/model and saves artifacts.
Importantly, the source column `yield_kg` is excluded from features to avoid leakage.

In [ ]:
from src.training_pipeline import run_training_pipeline

metrics = run_training_pipeline(data_path='data/raw/source_demo_crops_20260321.csv')
metrics

## 5) Inference drops leakage column at prediction time

At inference time we pass `target_column='yield_kg'` so it is removed if present.
Inference loads artifacts and runs `transform()` + `predict()` only (no refitting).

In [ ]:
from src.prediction_pipeline import run_prediction_pipeline

preds = run_prediction_pipeline(
    data_path='data/raw/source_demo_crops_20260321.csv',
    target_column='yield_kg',
    output_path='reports/predictions.csv',
)
preds.head()